# Value-Based Agents: DQN on Atari Breakout

CSCI 6353 · Topic 35. This notebook builds a Nature-style **Deep Q-Network** end to end and trains it to play Atari **Breakout** from raw pixels. It is the runnable companion to the lecture: the same architecture, the same two stabilizers (experience replay and a target network), and the same hyperparameters as the script that produced the agent shown in class. Read it top to bottom, then run the short sanity training at the end.

## 1. Environment setup

DQN on Atari needs a **GPU**. In Colab, choose **Runtime -> Change runtime type -> GPU** before running anything; on CPU a meaningful run would take days.

The cell below installs Gymnasium's Atari support. `gymnasium[atari]` pulls in the wrappers, `ale-py` is the Arcade Learning Environment, and `autorom` fetches the licensed Atari ROMs.

In [ ]:
# Run once per Colab session. Restart the runtime if pip asks you to.
!pip -q install "gymnasium[atari]" "ale-py" "autorom[accept-rom-license]"


In [ ]:
import os, csv, time, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)          # make the ALE/* environments importable

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
if dev == "cpu":
    print("WARNING: no GPU detected. Training will be extremely slow -- "
          "switch the Colab runtime to GPU.")


## 2. What the agent sees — Atari preprocessing

A raw Atari frame is $210\times160\times3$ pixels. Four preprocessing steps turn it into something a CNN can learn from, and Gymnasium's `AtariPreprocessing` wrapper does all of them for us:

- **Grayscale.** Colour is not needed to play, so we keep only luminance and drop two thirds of the data.
- **Downscale to $84\times84$.** Small enough to be cheap, large enough that the ball (a couple of pixels) survives.
- **Frame skip $k=4$ with max-pooling.** The agent acts every 4th frame and the action is repeated in between; each stored frame is the pixel-wise **max of two consecutive raw frames**, which fixes the Atari sprite flicker.
- **Episodic life (training only).** Losing one life is treated as terminal while training, which sharpens credit assignment. Evaluation uses true five-life episodes.

Then `FrameStackObservation` stacks the **last 4 frames** so the ball's *velocity* is part of the state — a single frame shows position but not direction, so one frame alone is not a Markov state.

Reward clipping (all positive rewards to $+1$, negative to $-1$) is applied later, in the training loop, so we can still log the true score.

In [ ]:
def make_env(train=True):
    # frameskip=1 here so the wrapper's frame_skip=4 is the only skipping,
    # and repeat_action_probability=0.0 turns off sticky actions.
    env = gym.make("ALE/Breakout-v5", frameskip=1, repeat_action_probability=0.0)
    env = gym.wrappers.AtariPreprocessing(
        env, noop_max=30, frame_skip=4, screen_size=84,
        terminal_on_life_loss=train,      # episodic life during training only
        grayscale_obs=True, scale_obs=False)
    return gym.wrappers.FrameStackObservation(env, 4)   # 4-frame stack

env = make_env(train=True)
N_ACT = env.action_space.n
s0, _ = env.reset(seed=SEED)
print("observation shape:", np.asarray(s0).shape, np.asarray(s0).dtype)  # (4, 84, 84) uint8
print("actions:", N_ACT, "->", env.unwrapped.get_action_meanings())      # NOOP, FIRE, RIGHT, LEFT


## 3. The Q-network

The state is a $4\times84\times84$ image stack, so the function approximator is a **CNN** — the Nature 2015 architecture: three convolutional layers, then two dense layers. It outputs **one value per action** in a single forward pass, so both $\max_a Q(s,a)$ (for the target) and $\arg\max_a Q(s,a)$ (for acting) cost one evaluation instead of one per action. For Breakout that is four outputs.

The three conv layers reduce $4\times84\times84$ to $64\times7\times7 = 3{,}136$ features; the dense layers map that to the action values. Pixels are divided by 255 inside `forward` so the buffer can stay `uint8`. The whole network is about **1.69 million** parameters.

In [ ]:
class QNet(nn.Module):
    """Nature 2015 DQN: three conv layers into two dense layers."""
    def __init__(self, n_act):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(), nn.Flatten())
        self.head = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, n_act))

    def forward(self, x):
        return self.head(self.conv(x.float() / 255.0))   # scale uint8 -> [0,1]

q = QNet(N_ACT).to(dev)
print(f"params: {sum(p.numel() for p in q.parameters()):,}")


## 4. Experience replay buffer

Every transition $(s, a, r, s', \text{done})$ goes into a large **circular buffer**, and training samples *random* mini-batches from it. This decorrelates the data — consecutive Atari frames are nearly identical, and training on them in order is unstable — and lets each transition be reused many times.

The states are stored as **`uint8`**, not float. This is not a micro-optimization: at $4\times84\times84$ bytes per state, kept for both $s$ and $s'$, `uint8` is what decides whether the buffer fits in memory at all. The Nature paper used a one-million-frame buffer; the course run used 200,000, limited by memory.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.cap = capacity
        self.S  = np.zeros((capacity, 4, 84, 84), np.uint8)   # state
        self.S2 = np.zeros((capacity, 4, 84, 84), np.uint8)   # next state
        self.A  = np.zeros(capacity, np.int64)                # action
        self.R  = np.zeros(capacity, np.float32)              # clipped reward
        self.D  = np.zeros(capacity, np.float32)              # done flag
        self.ptr = self.size = 0

    def add(self, s, a, r, s2, done):
        i = self.ptr
        self.S[i] = np.asarray(s); self.S2[i] = np.asarray(s2)
        self.A[i] = a; self.R[i] = np.sign(r); self.D[i] = float(done)  # reward clipping
        self.ptr = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, batch):
        idx = np.random.randint(0, self.size, batch)
        to = lambda arr: torch.from_numpy(arr[idx]).to(dev)
        return to(self.S), to(self.A), to(self.R), to(self.S2), to(self.D)


## 5. The epsilon-greedy policy

The agent explores by acting randomly with probability $\epsilon$ and greedily otherwise. $\epsilon$ is **annealed** from 1.0 (pure random, mostly filling the buffer) down to a small floor of 0.05 over the first 600,000 steps. A small floor is kept even at evaluation, because a purely greedy Breakout agent can get stuck never pressing FIRE and stall forever.

In [ ]:
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 600_000

def epsilon(step):
    return max(EPS_END, EPS_START - (EPS_START - EPS_END) * step / EPS_DECAY)

def act(net, state, eps):
    if random.random() < eps:
        return random.randrange(N_ACT)
    with torch.no_grad():
        t = torch.from_numpy(np.asarray(state)).unsqueeze(0).to(dev)
        return int(net(t).argmax(1).item())


## 6. Hyperparameters

These are the exact values from the course training script (`train_breakout.py`), which follow the Nature paper with the replay buffer scaled down to fit memory. The one number worth reading twice is `TARGET_EVERY`: the target network syncs every 2,000 **gradient steps**, not every 2,000 environment frames (Nature uses 10,000 gradient steps).

In [ ]:
GAMMA        = 0.99        # discount factor
BATCH        = 32          # mini-batch size
LR           = 1e-4        # Adam learning rate
BUFFER       = 200_000     # replay capacity (course run; see the memory note below)
LEARN_START  = 20_000      # steps of pure experience-gathering before training
TRAIN_EVERY  = 4           # one gradient step per 4 environment steps
TARGET_EVERY = 2_000       # gradient steps between target-network syncs


## 7. The training loop

This is Q-Learning with the two stabilizers wired in. Each iteration: act with the current $\epsilon$, store the transition, and — once the buffer has warmed up — take one gradient step every four env steps on a random mini-batch.

The target is
$$y = r + \gamma\,(1-\text{done})\,\max_{a'} Q_{\theta^-}(s', a'),$$
computed with the **frozen target network** $\theta^-$ under `torch.no_grad()`. We gather the online network's value for the action actually taken, and regress it onto $y$ with the **Huber loss** (`smooth_l1_loss`) rather than MSE, so a single wild target cannot produce a gradient large enough to wreck the network. Gradients are norm-clipped for the same reason. Every `TARGET_EVERY` gradient steps, the frozen copy is refreshed.

### A note on compute, and a short-run option

Be honest with yourself about the budget. A real DQN result takes **millions of environment steps and hours on a GPU** — the lecture's learning curve and the early/mid/late gifs came from a **2.5-million-step** run that took about **84 minutes on one cluster GPU**, and the buffer at the course's 200,000 capacity needs roughly **11 GB of RAM** (both $s$ and $s'$ as `uint8`), which is more than a free Colab instance has.

So the cell below is a **sanity check, not a trained agent.** It runs a few thousand steps with a small buffer just to confirm the whole pipeline steps, stores, samples, and back-propagates without error. Do **not** expect the score to rise in this budget. To reproduce the lecture's agent, set `DEMO = False`, raise `TOTAL_STEPS` to the millions, and run on a machine with enough memory (this is exactly what `train_breakout.py` does on the cluster).

In [ ]:
DEMO = True   # True: quick pipeline check.  False: real (long, GPU + lots of RAM) run.

if DEMO:
    TOTAL_STEPS = 5_000          # just enough to see the loop run
    buf = ReplayBuffer(20_000)   # small buffer so it fits in Colab RAM
    learn_start = 1_000
else:
    TOTAL_STEPS = 2_500_000      # course run; 10M frames after frame-skip
    buf = ReplayBuffer(BUFFER)   # ~11 GB of RAM -- needs a big machine
    learn_start = LEARN_START

# online network q (from section 3) + a frozen target copy
q_target = QNet(N_ACT).to(dev)
q_target.load_state_dict(q.state_dict())
opt = optim.Adam(q.parameters(), lr=LR)

s, _ = env.reset(seed=SEED)
ep_ret, ep, grad_steps, t0 = 0.0, 0, 0, time.time()
returns = []

for step in range(1, TOTAL_STEPS + 1):
    eps = epsilon(step)
    a = act(q, s, eps)
    s2, r, term, trunc, _ = env.step(a)
    done = term or trunc

    buf.add(s, a, r, s2, term)     # store; reward is clipped inside add()
    ep_ret += r
    s = s2
    if done:
        ep += 1
        returns.append(ep_ret)
        ep_ret = 0.0
        s, _ = env.reset()

    # one gradient step every TRAIN_EVERY env steps, once warmed up
    if step > learn_start and step % TRAIN_EVERY == 0:
        bs, ba, br, bs2, bd = buf.sample(BATCH)
        with torch.no_grad():                                   # frozen target net
            y = br + GAMMA * (1 - bd) * q_target(bs2).max(1)[0]
        pred = q(bs).gather(1, ba.unsqueeze(1)).squeeze(1)      # Q of the action taken
        loss = nn.functional.smooth_l1_loss(pred, y)           # Huber, not MSE
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(q.parameters(), 10.0)         # gradient clipping
        opt.step()
        grad_steps += 1
        if grad_steps % TARGET_EVERY == 0:                     # refresh frozen copy
            q_target.load_state_dict(q.state_dict())

    if step % 1_000 == 0:
        last = np.mean(returns[-20:]) if returns else float("nan")
        print(f"step {step:>7,}  eps {eps:.3f}  grad_steps {grad_steps:>6,}  "
              f"lives {ep:>4}  last20 {last:5.1f}  ({(time.time()-t0)/60:.1f} min)")

print("done. this was a pipeline check -- do not read anything into the score.")


## Recap

- **DQN = Q-Learning + a convolutional network + experience replay + a target network.** The last two exist only to make the first two stable enough to work.
- The agent sees **raw pixels**: grayscale, $84\times84$, **four frames stacked** so velocity is part of the state and the problem stays Markov.
- The network outputs **all action values at once**; ours has about **1.69 million** parameters.
- The loss is a regression onto $y = r + \gamma\max_{a'} Q_{\theta^-}(s', a')$ with **Huber** instead of squared error.
- The details are load-bearing: reward clipping, frame skip, $\epsilon$ annealing, `uint8` storage, gradient clipping, and a target sync counted in **gradient steps**, not frames.
- The course's own 2.5-million-step run learned Breakout from pixels in **84 minutes on one GPU**, improving from **1.9** points per game at the 100k checkpoint to **161** at 2.5M — still short of the Nature paper's **401** on a fifth of the training budget, and without discovering the tunnelling strategy.